# Phase 0 — Temporal parity QA

Audits year coverage across weather layers before Phase 1.


- **10 km grid** (`NY_State_10km_*.csv`) — primary solar + wind source (2018–2025)
- **County solar / wind** — documented gaps; county wind API gap-fill deferred

**Output:** `data/processed/temporal_parity_report.json`

In [ ]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import yaml

ROOT = Path('../..').resolve()
CONFIG_PATH = ROOT / 'config' / 'paths.yaml'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'temporal_parity_report.json'

with CONFIG_PATH.open(encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

YEAR_PATTERN = re.compile(r'(\d{4})')
print('Repo root:', ROOT)
print('Target years:', CONFIG['weather']['grid_years'])

In [2]:
def years_in_dir(directory: Path, glob_pattern: str) -> list[int]:
    if not directory.exists():
        return []
    years = set()
    for path in directory.glob(glob_pattern):
        match = YEAR_PATTERN.search(path.stem)
        if match:
            years.add(int(match.group(1)))
    return sorted(years)


def build_report(config: dict) -> dict:
    raw = config['data']['raw']
    grid_dir = ROOT / raw['grid_weather_dir']
    solar_dir = ROOT / raw['county_solar_dir']
    wind_dir = ROOT / raw['county_wind_dir']
    target_years = config['weather']['grid_years']

    grid_years = years_in_dir(grid_dir, 'NY_State_10km_*.csv')
    solar_years = years_in_dir(solar_dir, 'ny_county_solar_hourly_*.csv')
    wind_years = years_in_dir(wind_dir, 'ny_county_wind_hourly_*.csv')

    return {
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'wind_source_policy': config['weather']['wind_source'],
        'target_years': target_years,
        'layers': {
            'grid_10km': {
                'path': str(grid_dir.relative_to(ROOT)),
                'years_found': grid_years,
                'years_missing': sorted(set(target_years) - set(grid_years)),
                'ready_for_phase1': grid_years == target_years,
            },
            'county_solar': {
                'path': str(solar_dir.relative_to(ROOT)),
                'years_found': solar_years,
                'years_missing': sorted(set(target_years) - set(solar_years)),
            },
            'county_wind': {
                'path': str(wind_dir.relative_to(ROOT)),
                'years_found': wind_years,
                'years_missing': sorted(set(target_years) - set(wind_years)),
                'status': 'incomplete_api_gap_fill_deferred',
            },
        },
        'phase0_gate': {
            'county_wind_parity': False,
            'grid_wind_fallback_accepted': True,
            'proceed_to_phase1': grid_years == target_years,
            'note': (
                'County wind 2021-2025 gap-fill deferred (Open-Meteo unavailable). '
                'Phase 1 uses 10 km grid wind_speed_10m for aligned 2018-2025 features.'
            ),
        },
    }

In [ ]:
report = build_report(CONFIG)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')

print(json.dumps(report, indent=2))
print(f'\nWrote {OUTPUT_PATH}')

if report['phase0_gate']['proceed_to_phase1']:
    print('\n✅ Grid weather complete — proceed to 01_ingest_weather.ipynb')
else:
    print('\n⚠️ Missing grid years — fix raw 10 km files before Phase 1.')